# Hotel Room Type Preprocessing

Notebook này dùng để làm sạch và chuẩn hóa dữ liệu loại phòng (room type) từ file `hotel_room_types.csv`.

## Mục tiêu:
- Trích xuất **chỉ tên tiếng Anh** của loại phòng
- Bỏ phần tiếng Việt
- Loại bỏ các chi tiết như "with view", "with balcony", "city view", etc.
- Chỉ giữ lại loại phòng cơ bản: Standard, Deluxe, Superior, Suite, Executive, Premium, Studio, Apartment, Twin, Double, King, Single, Triple, Quadruple, Family, etc.


## 1. Import thư viện và đọc dữ liệu


In [4]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)


ModuleNotFoundError: No module named 'pandas'

In [ ]:
file_path = '../../data/processed/hotel_room_types.csv'
df = pd.read_csv(file_path)

print(f"Tổng số dòng: {len(df)}")
print(f"Số khách sạn: {df['hotel_id'].nunique()}")
print(f"Số loại phòng khác nhau: {df['room_room_type_name'].nunique()}")
print(f"\nMẫu dữ liệu đầu tiên:")
df.head(20)


## 2. Hàm trích xuất tên tiếng Anh và làm sạch


In [ ]:
def extract_english_room_type(text):
    """
    Trích xuất tên tiếng Anh của loại phòng từ text
    - Nếu có phần trong ngoặc đơn, lấy phần đó
    - Nếu không có ngoặc, kiểm tra xem có phải tiếng Anh không
    - Loại bỏ các chi tiết như "with view", "with balcony", etc.
    """
    if pd.isna(text):
        return np.nan
    
    text = str(text).strip()
    
    # Tìm phần trong ngoặc đơn (thường là tiếng Anh)
    match = re.search(r'\(([^)]+)\)', text)
    if match:
        english_part = match.group(1).strip()
    else:
        # Nếu không có ngoặc, kiểm tra xem có chứa ký tự tiếng Việt không
        has_vietnamese = bool(re.search(r'[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', text, re.IGNORECASE))
        if not has_vietnamese:
            english_part = text
        else:
            return np.nan
    
    if not english_part or english_part.lower() == 'nan':
        return np.nan
    
    # Loại bỏ các chi tiết không cần thiết
    words_to_remove = [
        r'\bwith\s+\w+\s+view\b',
        r'\bcity\s+view\b',
        r'\bgarden\s+view\b',
        r'\bpool\s+view\b',
        r'\bsea\s+view\b',
        r'\bocean\s+view\b',
        r'\bwith\s+balcony\b',
        r'\bbalcony\b',
        r'\bwith\s+bathtub\b',
        r'\bwith\s+spa\s+bath\b',
        r'\bground\s+floor\b',
        r'\bsky\s+window\b',
        r'\bnon-smoking\b',
        r'\bsmoke\s+free\b',
        r'\broom\s+only\b',
        r'\bfor\s+\d+\s+people\b',
        r'\bfor\s+\d+\s+guests\b',
    ]
    
    cleaned = english_part
    for pattern in words_to_remove:
        cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
    
    cleaned = re.sub(r'\bwith\b', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    cleaned = re.sub(r'^[,.\s]+|[,.\s]+$', '', cleaned)
    
    if len(cleaned.strip()) < 2:
        return np.nan
    
    return cleaned.strip()


def simplify_room_type_name(text):
    """
    Đơn giản hóa tên loại phòng - chỉ giữ lại loại phòng cơ bản
    """
    if pd.isna(text):
        return text
    
    text = str(text).strip()
    
    # Loại bỏ số phòng ngủ ở đầu
    text = re.sub(r'^\d+\s*bedroom\s+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^one\s+bedroom\s+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^two\s+bedroom\s+', '', text, flags=re.IGNORECASE)
    
    # Loại bỏ từ "Room" ở cuối
    text = re.sub(r'\s+room$', '', text, flags=re.IGNORECASE)
    
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Test hàm
test_cases = [
    "Phòng Deluxe Có Giường Cỡ King (Deluxe King Room)",
    "Phòng Tiêu Chuẩn (Standard Room)",
    "Deluxe hướng thành phố (Deluxe City View)",
    "Suite cao cấp (Premium Suite)",
    "TWIN PREMIUM DELUXE TWIN",
    "Studio Deluxe with Bathtub",
    "Phòng Gia Đình có ban công (Family Room with Balcony)",
    "1 Bedroom Deluxe",
    "Superior 2 giường (Superior Twin)",
    "Phòng Đôi (Double Room)",
]

print("Test hàm trích xuất:")
for test in test_cases:
    english = extract_english_room_type(test)
    simplified = simplify_room_type_name(english) if pd.notna(english) else np.nan
    print(f"Original: {test!r}")
    print(f"English:  {english!r}")
    print(f"Simplified: {simplified!r}")
    print()


## 3. Áp dụng trích xuất và làm sạch


In [ ]:
df_processed = df.copy()

print("Đang trích xuất tên tiếng Anh...")
df_processed['room_type_english'] = df_processed['room_room_type_name'].apply(extract_english_room_type)
df_processed['room_type_english'] = df_processed['room_type_english'].apply(simplify_room_type_name)

print(f"Số dòng: {len(df_processed)}")
print(f"Số dòng có tên tiếng Anh: {df_processed['room_type_english'].notna().sum()}")
print(f"Số dòng không có tên tiếng Anh: {df_processed['room_type_english'].isna().sum()}")


In [ ]:
# Chuẩn hóa viết hoa
def standardize_capitalization(text):
    if pd.isna(text):
        return text
    text = str(text).strip()
    if text.isupper() and len(text) > 1:
        return text.title()
    return text

df_processed['room_type_english'] = df_processed['room_type_english'].apply(standardize_capitalization)
df_processed['room_type_english'] = df_processed['room_type_english'].str.strip() if df_processed['room_type_english'].notna().any() else df_processed['room_type_english']

print("Đã chuẩn hóa viết hoa")
print(f"\nTop 30 loại phòng xuất hiện nhiều nhất:")
top_types = df_processed['room_type_english'].value_counts().head(30)
print(top_types)


## 4. Lọc và lưu kết quả


In [ ]:
# Loại bỏ các dòng không có tên tiếng Anh
df_final = df_processed[df_processed['room_type_english'].notna()].copy()
df_final = df_final[df_final['room_type_english'].str.strip() != ''].copy()

print(f"Số dòng trước khi lọc: {len(df_processed)}")
print(f"Số dòng sau khi lọc: {len(df_final)}")
print(f"Đã loại bỏ: {len(df_processed) - len(df_final)} dòng")

df_output = df_final[['hotel_id', 'room_type_english', 'room_type_id']].copy()
df_output.rename(columns={'room_type_english': 'room_room_type_name'}, inplace=True)

print(f"\nThông tin dataset cuối cùng:")
print(f"- Tổng số dòng: {len(df_output)}")
print(f"- Số khách sạn: {df_output['hotel_id'].nunique()}")
print(f"- Số loại phòng unique: {df_output['room_room_type_name'].nunique()}")
print(f"- Số giá trị null: {df_output.isnull().sum().sum()}")

print("\nMẫu dữ liệu cuối cùng:")
df_output.head(30)


In [ ]:
# Lưu file đã được làm sạch
output_path = 'hotel_room_types_cleaned.csv'
df_output.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Đã lưu file đã làm sạch tại: {output_path}")
print(f"Tổng số dòng đã lưu: {len(df_output)}")


## 5. Tóm tắt

### Kết quả:
- ✅ Đã trích xuất **chỉ tên tiếng Anh** của loại phòng
- ✅ Đã loại bỏ phần tiếng Việt
- ✅ Đã loại bỏ các chi tiết như "with view", "with balcony", "city view", etc.
- ✅ Chỉ giữ lại loại phòng cơ bản

### File output:
- `hotel_room_types_cleaned.csv`: File đã được làm sạch (chỉ tên tiếng Anh)
